In [ ]:
!cd /kaggle/working/activeqa_final && git pull && git log --oneline -1

In [ ]:
!cd /kaggle/working/activeqa_final && python scripts/kaggle_run_all.py --from-base

In [ ]:
import sys, json, random
ROOT = "/kaggle/working/activeqa_final"
sys.path.insert(0, ROOT + "/src")
import torch
from activeqa.reformulator import Reformulator
from activeqa.qa_model import TeluguQA
from activeqa.preprocessing import normalize
from activeqa.rewards import token_f1

BASE = ROOT + "/models/MultiIndicParaphraseGenerationSS"
ADAPTER = ROOT + "/models/indicbartss-reformulator-lora-rl/adapter"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); random.seed(42)

ref = Reformulator(BASE, cache_dir=ROOT + "/models", device=device,
                   rank=32, alpha=64, adapter_path=ADAPTER)
qa = TeluguQA("l3cube-pune/telugu-question-answering-squad-bert",
              cache_dir=ROOT + "/models", device=device)

rows = [json.loads(l) for l in open(ROOT + "/data/indic_rag_te/dev.jsonl", encoding="utf-8")][:3]
for row in rows:
    q = normalize(row["question"]); ctx = row["context"]; gold = row["answers"]
    ans_q, _, score_q = qa.answer_with_score(q, ctx)
    acts = ref.sample_actions(q, n=6, max_input=96, max_new=64, do_sample=True)
    valid = [a for a in acts if a["valid"]]
    print("=" * 78)
    print("Q :", q)
    print("   gold:", gold)
    print(f"   ANSWER(original) [{score_q:.2f}]: {ans_q}   f1={token_f1(ans_q, gold):.2f}")
    if not valid:
        print("   (all candidates invalid -> rewritten question would not be used)")
    for i, a in enumerate(valid, 1):
        ans, _, s = qa.answer_with_score(a["candidate"], ctx)
        print(f"   cand {i} (valid): {a['candidate']}")
        print(f"      ANSWER [{s:.2f}]: {ans}   f1={token_f1(ans, gold):.2f}")
print("=" * 78)
print("valid-rate on last question:", sum(a['valid'] for a in acts), "/", len(acts))
